# 02. PDM и delta-sigma первого/второго порядка

PDM и one-bit delta-sigma в этом проекте дают один выходной бит на один входной сэмпл. Поэтому для спектров здесь используется частота данных `f_data`, а не `f_clk` PWM.

Важное отличие от PWM: нет треугольной несущей. Шум и пульсации определяются состояниями аккумулятора/петли ошибки.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if (HERE / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE
elif (HERE / "tutorials" / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE / "tutorials"
else:
    raise RuntimeError("Run this notebook from the repository root or from the tutorials folder")

path_text = str(TUTORIAL_DIR)
if path_text not in sys.path:
    sys.path.insert(0, path_text)

import matplotlib.pyplot as plt
import numpy as np

from tutorial_helpers import (
    configure_plots,
    grouped_fifo_channel_waveforms,
    load_pwm_lab,
    plot_bitstream,
    plot_channel_stack,
    plot_moving_average_reconstruction,
    plot_pwm_carrier_output,
    plot_spectra,
    print_peak_table,
    pwm_kind2_channel_waveforms,
    show_grouped_mapping,
    time_us,
)

pl = load_pwm_lab()
configure_plots()

## Нормированный вход `[0, 1]`

In [ ]:
f_data = 2e6
f_signal = 20e3
n_samples = 8192

_, x = pl.sine_samples(freq=f_signal, sample_rate=f_data, n_samples=n_samples, amplitude=0.85)

pdm1 = pl.pdm_first_order(x)
pdm2 = pl.pdm_second_order(x)
ds1 = pl.delta_sigma_first_order(x)
ds2 = pl.delta_sigma_second_order(x)

## Временные реализации

Первый порядок обычно дает более регулярное распределение импульсов. Второй порядок сильнее меняет структуру ошибки, что лучше видно в спектре.

In [ ]:
plot_bitstream(x, pdm1, sample_rate=f_data, max_points=420, title="PDM first order");
plot_bitstream(x, pdm2, sample_rate=f_data, max_points=420, title="PDM second order");
plot_bitstream(x, ds2, sample_rate=f_data, max_points=420, title="Delta-sigma second order");

## Грубая реконструкция через moving average

Это быстрый способ проверить, что плотность импульсов повторяет входной сигнал. Для настоящей аудио/силовой оценки нужен свой фильтр и модель нагрузки.

In [ ]:
plot_moving_average_reconstruction(
    x,
    pdm2,
    input_sample_rate=f_data,
    modulated_sample_rate=f_data,
    average_factor=64,
    title="PDM second-order moving average",
);

plot_moving_average_reconstruction(
    x,
    ds2,
    input_sample_rate=f_data,
    modulated_sample_rate=f_data,
    average_factor=64,
    title="Delta-sigma second-order moving average",
);

## Спектры первого и второго порядка

In [ ]:
plot_spectra(
    {
        "PDM 1": pdm1,
        "PDM 2": pdm2,
        "Delta-sigma 1": ds1,
        "Delta-sigma 2": ds2,
    },
    sample_rate=f_data,
    f_max=600e3,
    f_scale=1e3,
    f_unit="kHz",
    title="PDM and delta-sigma spectra",
);

print_peak_table(
    {"PDM 1": pdm1, "PDM 2": pdm2, "DS 1": ds1, "DS 2": ds2},
    sample_rate=f_data,
    f_min=1.0,
    f_max=600e3,
    f_scale=1e3,
    f_unit="kHz",
)

## Signed и bipolar варианты

Для signed delta-sigma поток принимает значения `-1` и `+1`. Bipolar-вариант разделяет положительную и отрицательную ветви, а полезный сигнал смотрится как `positive - negative`.

In [ ]:
_, x_signed = pl.sine_signed(freq=f_signal, sample_rate=f_data, n_samples=n_samples, amplitude=0.8)

ds_signed = pl.delta_sigma_second_order_signed(x_signed)
ds_bipolar = pl.delta_sigma_second_order_bipolar(x_signed)

plot_bitstream(x_signed, ds_signed, sample_rate=f_data, max_points=420, title="Signed delta-sigma second order");

n = 420
t = time_us(f_data, n)
fig, axes = plt.subplots(4, 1, figsize=(11, 6.4), sharex=True, constrained_layout=True)
axes[0].plot(t, x_signed[:n], color="C1")
axes[0].set_ylabel("input")
axes[0].set_title("Bipolar delta-sigma branches")
axes[1].step(t, ds_bipolar.positive[:n], where="post", color="C0")
axes[1].set_ylabel("positive")
axes[2].step(t, ds_bipolar.negative[:n], where="post", color="C3")
axes[2].set_ylabel("negative")
axes[3].step(t, ds_bipolar.differential[:n], where="post", color="C2")
axes[3].set_ylabel("diff")
axes[3].set_xlabel("time, us");